# VoiceSecure 훈련 노트북 (SDK Evaluator 파이프라인)

**실행 순서대로 셀을 실행하세요.**

### 사전 준비
- 런타임 유형 = **GPU (T4 이상)**
- 다음 중 하나로 데이터 준비:
  - **AIHub 014 다화자 음성합성 데이터** — Step 2-B의 aihubshell로 직접 다운로드
  - **KSS 데이터셋** — Google Drive에 미리 업로드

### 파이프라인 (의도된 매핑)
```
원본 음성
  └─▶ RLAgent.act()             — state(36-dim) → 노이즈 action(257×100)
  └─▶ PsychoacousticMasker      — 심리음향 임계치로 clamp
  └─▶ Mixer                     — STFT 도메인 합성 → 변조 음성
       │
       ├─ SpeakerEvaluator       (WavLM-SV + CAM++)      → sv_score
       ├─ TTSEvaluator           (XTTS, 10 ep마다)        → tts_score
       └─ ASREvaluator           (wav2vec2-xlsr-korean)   → asr_cer
            └▶ RewardFunction (alpha·sv + beta·tts - lambda·max(0, cer-tau))
                  └▶ PPO update
```

기존 CosyVoice + ECAPA 직접 호출 흐름은 제거됨. 모두 SDK Evaluator 통과.


## Step 1. GPU 확인

In [ ]:
import torch, sys

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: GPU 없음 — 런타임 > 런타임 유형 변경 > GPU 설정 필요')


## Step 2. Google Drive 마운트 + 데이터 경로 설정

아래 셀 실행 후 Drive 연결 허용을 눌러주세요.

**DATA_FORMAT** (kss / aihub) 과 **DATA_DIR**을 본인 데이터에 맞게 수정하세요.

> Drive에 안 올리고 Colab에 직접 받고 싶으면 → **Step 2-B (aihubshell)** 이용


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ─────────────────────────────────────────────────────────────────────
# 데이터 경로 설정 — 사용하는 데이터셋에 따라 한쪽만 채우세요
# ─────────────────────────────────────────────────────────────────────

# (A) AIHub 014 다화자 음성합성 데이터 사용 시
DATA_FORMAT = 'aihub'
DATA_DIR    = '/content/drive/MyDrive/aihub_014'   # 원천데이터/, 라벨링데이터/ 의 부모 폴더

# AIHub 옵션 (None=전체)
MAX_SPEAKERS           = None    # 예: 10 → 처음 10명만 사용
MAX_FILES_PER_SPEAKER  = 500     # 화자당 500개로 균형 → 학습 시간 단축

# (B) KSS 데이터셋 사용 시 (이쪽 쓰면 위 두 줄 주석 처리하고 아래 활성화)
# DATA_FORMAT = 'kss'
# DATA_DIR    = '/content/drive/MyDrive/kss'

print(f'데이터 포맷: {DATA_FORMAT}')
print(f'데이터 경로: {DATA_DIR}')


## Step 2-B. (선택) aihubshell로 AIHub 데이터 직접 다운로드

Drive 업로드 없이 **Colab에서 AIHub에서 직접 다운로드**.

### 사전 조건
1. AI 허브 회원가입 + 본인인증
2. **014. 다화자 음성합성 데이터** 사용 신청 → 승인 완료
3. **마이페이지 → API 활용**에서 API 키 발급

### KSS 또는 이미 Drive에 데이터가 있으면 이 Step 건너뛰기.


In [ ]:
# ─── aihubshell 설치 + AIHub 데이터 다운로드 (선택) ───────────────────
AIHUB_API_KEY = ''   # ← AIHub 마이페이지 → API 활용에서 발급받은 키

if AIHUB_API_KEY:
    !curl -s -o /usr/local/bin/aihubshell https://api.aihub.or.kr/api/aihubshell.do
    !chmod +x /usr/local/bin/aihubshell
    print('aihubshell 설치 완료')

    print('=== 데이터셋 검색 (datasetkey 확인) ===')
    !aihubshell -mode l | grep -E "다화자|음성합성" | head -20
    print()
    print('↑ "다화자 음성합성 데이터"의 datasetkey 숫자를 확인하세요.')
else:
    print('AIHUB_API_KEY를 먼저 입력하세요 (https://aihub.or.kr → 마이페이지 → API 활용)')


In [ ]:
# ─── (datasetkey 확인 후) 파일 목록 + 다운로드 ───────────────────────
DATASET_KEY  = ''   # ← 위 셀에서 확인한 숫자
FILE_KEYS    = ''   # ← 다운받을 filekey들 콤마 구분
DOWNLOAD_DIR = '/content/aihub_014'

if AIHUB_API_KEY and DATASET_KEY and not FILE_KEYS:
    print(f'=== datasetkey {DATASET_KEY} 의 파일 구조 ===')
    !aihubshell -mode l -datasetkey {DATASET_KEY}
    print('↑ TL/TS 의 filekey(맨 우측 숫자)를 콤마로 묶어 FILE_KEYS에 입력 후 다시 실행')

elif AIHUB_API_KEY and DATASET_KEY and FILE_KEYS:
    import os
    os.makedirs(DOWNLOAD_DIR, exist_ok=True)
    %cd {DOWNLOAD_DIR}
    !aihubshell -mode d -datasetkey {DATASET_KEY} -filekey {FILE_KEYS} -aihubapikey '{AIHUB_API_KEY}'
    !ls -la {DOWNLOAD_DIR}
    print('↑ 데이터셋 폴더명 확인 후 Step 2의 DATA_DIR을 수정하세요')

else:
    print('AIHUB_API_KEY와 DATASET_KEY를 먼저 설정하세요.')


## Step 3. VoiceSecure SDK 설치

GitHub에서 SDK를 clone하고 editable 모드로 설치합니다.
이미 있으면 `git pull`로 최신화.


In [ ]:
import os

SDK_ROOT = '/content/voicesecure-sdk'

if not os.path.exists(SDK_ROOT):
    !git clone https://github.com/VoiceSecureHoseo/voicesecure-sdk.git {SDK_ROOT}
else:
    !git -C {SDK_ROOT} pull
    print('SDK 업데이트 완료')

%cd {SDK_ROOT}
!pip install -q -e .
print('SDK 설치 완료')


## Step 4. 추가 의존성 설치

- **coqui-tts** — XTTS v2 (TTSEvaluator 백엔드). XTTS는 CPML 라이선스이므로 `COQUI_TOS_AGREED=1` 환경변수로 동의 표시.
- `transformers`, `scipy`, `soundfile`, `onnxruntime`, `tensorboard` 는 SDK pyproject 의존성이라 Step 3에서 이미 설치됨.

CosyVoice/SpeechBrain/ECAPA 의존성은 새 파이프라인에서 **사용 안 함**.


In [ ]:
!pip install -q coqui-tts

import os
os.environ['COQUI_TOS_AGREED'] = '1'
print('[OK] coqui-tts installed, XTTS TOS agreed')


## Step 5. 어댑터 로드 검증

4개 어댑터를 순서대로 로드합니다.
첫 실행 시 모델 가중치 다운로드:
- WavLM-SV: 380 MB (~1분)
- CAM++ ONNX: 30 MB (~30초)
- XTTS v2: 1.8 GB (~3분)
- wav2vec2-xlsr-korean: 1.2 GB (~2분)

총 약 7-10분.


In [ ]:
import time

from voicesecure.evaluators.adapters.wavlm_sv import WavLMSVAdapter
from voicesecure.evaluators.adapters.campplus import CAMPlusAdapter
from voicesecure.evaluators.adapters.xtts import XTTSAdapter
from voicesecure.evaluators.adapters.wav2vec2_asr import Wav2Vec2KoreanAdapter

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

t0 = time.time(); wavlm = WavLMSVAdapter(device=device);                   print(f'[load] WavLM-SV    {time.time()-t0:5.1f}s')
t0 = time.time(); cam   = CAMPlusAdapter(device=device);                   print(f'[load] CAM++        {time.time()-t0:5.1f}s')
t0 = time.time(); xtts  = XTTSAdapter();                                    print(f'[load] XTTS v2      {time.time()-t0:5.1f}s')
t0 = time.time(); asr   = Wav2Vec2KoreanAdapter(device=device);             print(f'[load] wav2vec2-ko  {time.time()-t0:5.1f}s')
print('어댑터 4종 로드 완료')


## Step 6. 빠른 동작 확인 (1 에피소드)

훈련 전 SDK Evaluator 체인이 정상 동작하는지 확인.
합성 오디오(1초)로 act → clamp → mix → SpeakerEvaluator → ASREvaluator → TTSEvaluator → RewardFunction.


In [ ]:
import numpy as np

from voicesecure.evaluators import SpeakerEvaluator, TTSEvaluator, ASREvaluator
from voicesecure.modulation.masker import PsychoacousticMasker
from voicesecure.modulation.mixer import Mixer
from voicesecure.reward.function import RewardFunction
from voicesecure.rl.agent import RLAgent
from voicesecure.types import SAMPLE_RATE
from train import emb_dist_normalizer    # cosine-distance → [0,1] 스케일 함수

# 합성 테스트 오디오 (1초)
rng = np.random.default_rng(42)
t = np.arange(SAMPLE_RATE, dtype=np.float32) / SAMPLE_RATE
test_audio = (
    0.3 * np.sin(2 * np.pi * 200 * t)
    + 0.2 * np.sin(2 * np.pi * 800 * t)
    + 0.02 * rng.standard_normal(SAMPLE_RATE).astype(np.float32)
).astype(np.float32)
test_text = '안녕하세요'

# Evaluator 조합 (의도된 매핑)
sv_eval  = SpeakerEvaluator(wavlm_model=wavlm, cam_model=cam, normalizer=emb_dist_normalizer)
tts_eval = TTSEvaluator(xtts_model=xtts, speaker_model=wavlm,
                        normalizer=emb_dist_normalizer, sampling_interval=10)
asr_eval = ASREvaluator(asr_model=asr)
reward_fn = RewardFunction()   # SDK default: alpha=0.6, beta=0.4, lambda_asr=1.0, cer_threshold=0.3

# 모듈
agent  = RLAgent()
masker = PsychoacousticMasker()
mixer  = Mixer()

# 1 에피소드
state, action, log_prob, value, freq_pattern, time_gate = agent.act(test_audio)
safe_noise = masker.clamp(test_audio, action)
modified   = mixer.mix(test_audio, safe_noise)

sv_out  = sv_eval.evaluate(test_audio, modified)
tts_out = tts_eval.evaluate(test_audio, modified)
asr_out = asr_eval.evaluate(test_audio, modified, original_text=test_text)

components = {
    'sv_score':  float(np.clip(sv_out.score, 0, 1)),
    'tts_score': float(np.clip(tts_out.score, 0, 1)),
    'asr_cer':   float(np.clip(asr_out.raw_metric, 0, 1)),
}
reward = reward_fn.compute(components)

print(f'action shape : {tuple(action.shape)}  (기대: (257, 100))')
print(f'sv_score     : {sv_out.score:.4f}  (raw={sv_out.raw_metric:.4f})')
print(f'tts_score    : {tts_out.score:.4f}  (raw={tts_out.raw_metric:.4f})')
print(f'asr_cer      : {asr_out.raw_metric:.4f}  (score={asr_out.score:.4f})')
print(f'reward       : {reward:.4f}')
print('파이프라인 정상 동작 확인')


## Step 7. 훈련 실행

### 새 train.py CLI 인자
| 인자 | 기본값 | 설명 |
|---|---|---|
| `--data_format` | `kss` | kss / aihub |
| `--max_speakers` | None | (aihub) 처음 N명만 |
| `--max_files_per_speaker` | None | (aihub) 화자당 N개 |
| `--epochs` | 3 | 1 에폭 = 데이터셋 전체 |
| `--checkpoint_interval` | 1000 | 에피소드 단위 |
| `--lr` | 3e-4 | PPO Adam lr |
| `--resume` | None | 이어서 훈련 |
| `--xtts_model_dir` | None (자동 탐색) | XTTS 모델 경로 |
| `--use_tts / --no-use-tts` | True | TTS 평가 끄기 |
| `--use_asr / --no-use-asr` | True | ASR 평가 끄기 |
| `--tts_eval_interval` | 10 | TTS 평가 주기 |
| `--reward_alpha / --reward_beta / --reward_lambda_asr / --reward_cer_threshold` | 0.6 / 0.4 / 1.0 / 0.3 | reward 가중치 |

학습이 느리면 `--no-use-asr` 또는 `--tts_eval_interval 50` 으로 가볍게 시작 가능.


In [ ]:
CHECKPOINT_DIR = f'{SDK_ROOT}/checkpoints'

if DATA_FORMAT == 'aihub':
    extra_args = '--data_format aihub'
    if MAX_SPEAKERS is not None:
        extra_args += f' --max_speakers {MAX_SPEAKERS}'
    if MAX_FILES_PER_SPEAKER is not None:
        extra_args += f' --max_files_per_speaker {MAX_FILES_PER_SPEAKER}'
else:
    extra_args = '--data_format kss'

!python {SDK_ROOT}/train.py \
    --data_dir       {DATA_DIR} \
    --epochs         3 \
    --checkpoint_dir {CHECKPOINT_DIR} \
    {extra_args}


### 이어서 훈련 (런타임 재연결 후)

Colab 런타임이 끊겼다 재연결 시 아래 셀로 재개.


In [ ]:
RESUME_CKPT = f'{CHECKPOINT_DIR}/best.pt'   # 또는 episode_N.pt

if DATA_FORMAT == 'aihub':
    extra_args = '--data_format aihub'
    if MAX_SPEAKERS is not None:
        extra_args += f' --max_speakers {MAX_SPEAKERS}'
    if MAX_FILES_PER_SPEAKER is not None:
        extra_args += f' --max_files_per_speaker {MAX_FILES_PER_SPEAKER}'
else:
    extra_args = '--data_format kss'

!python {SDK_ROOT}/train.py \
    --data_dir       {DATA_DIR} \
    --epochs         3 \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --resume         {RESUME_CKPT} \
    {extra_args}


## Step 9. TensorBoard 로그 확인

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {CHECKPOINT_DIR}/logs


## Step 10. 방어 효과 평가 (XTTS 클로닝)

훈련된 에이전트로 데이터셋 파일 하나를 변조한 뒤,
XTTS 클로닝 전/후 WavLM 화자 임베딩 거리를 비교.

**목표 지표**: cosine distance > 0.3 (SIM < 0.25 논문 기준)


In [ ]:
import soundfile as sf
import json as json_mod
import numpy as np
from math import gcd
from pathlib import Path
from scipy.signal import resample_poly

from voicesecure.evaluators.base import cosine_distance

def load_audio(path, sr=16000):
    data, orig_sr = sf.read(str(path), dtype='float32', always_2d=True)
    audio = data.mean(axis=1).astype(np.float32)
    if orig_sr != sr:
        g = gcd(sr, orig_sr)
        audio = resample_poly(audio, sr // g, orig_sr // g).astype(np.float32)
    return np.clip(audio, -1.0, 1.0)

# 평가용 agent (best 체크포인트 로드)
agent_eval = RLAgent()
agent_eval.load(f'{CHECKPOINT_DIR}/best.pt')

# 평가할 파일 선택 (데이터 포맷에 따라 다르게)
if DATA_FORMAT == 'aihub':
    test_wav = next((Path(DATA_DIR) / '원천데이터').rglob('*.wav'))
    json_path = next((Path(DATA_DIR) / '라벨링데이터').rglob(f'{test_wav.stem}.json'))
    with open(json_path, encoding='utf-8') as f:
        test_text = json_mod.load(f)['전사정보']['OrgLabelText']
else:
    test_wav = next(Path(DATA_DIR).rglob('*.wav'))
    test_text = '안녕하세요'    # KSS는 Labels.txt 파싱 필요 — placeholder

print('평가 파일:', test_wav)
print('텍스트   :', test_text)

original = load_audio(test_wav)
orig_emb = wavlm.extract_embedding(original)

# 변조
_, action, _, _, _, _ = agent_eval.act(original, deterministic=True)
safe_noise = masker.clamp(original, action)
modified   = mixer.mix(original, safe_noise)

# (a) 변조 음성 직접 거리
mod_emb     = wavlm.extract_embedding(modified)
direct_dist = cosine_distance(orig_emb, mod_emb)

# (b) 클로닝 후 거리 (실제 공격 시나리오)
# 평가용으로 clone_text를 일시 변경
import dataclasses
xtts.clone_text = test_text
cloned    = xtts.clone(modified)
clone_emb = wavlm.extract_embedding(cloned)
clone_dist = cosine_distance(orig_emb, clone_emb)

print()
print('--- 방어 효과 평가 ---')
print(f'직접 WavLM 거리 (원본 vs 변조): {direct_dist:.4f}')
print(f'클론 WavLM 거리 (원본 vs 클론): {clone_dist:.4f}   ← 핵심 지표')
print(f'목표:                            > 0.30')
print(f'달성 여부:                       {"성공" if clone_dist > 0.30 else "미달"}')


## Step 11. 체크포인트 Drive 백업

`/content` 안의 파일은 런타임 종료 시 삭제됩니다.
훈련 후 아래 셀로 Drive에 백업.


In [ ]:
import shutil

BACKUP_DIR = '/content/drive/MyDrive/voicesecure_checkpoints'

shutil.copytree(CHECKPOINT_DIR, BACKUP_DIR, dirs_exist_ok=True)
print(f'백업 완료: {BACKUP_DIR}')
